<style>
    .jp-RenderedHTMLCommon {
        font-family: "Times New Roman", Times, serif !important;
        font-size: 16px !important;
    }
</style>

### **1. Постановка задачи**

Рассмотрим набор различных точек на отрезке $[a, b]$: $x_0 < x_1 < \dots < x_n$, в
которых заданы значения функции $f(x)$ так, что $f_i = f(x), \quad і = 0,1, \dots, n$. Требуется востановить значение $f(x)$ и в других точках $х \in [a, b]$.

Вариант исходных данных:

Для построения таблицы $(х_i, f_i)$ дано:

$$[a, b] = [\alpha_j, 1 + \alpha_j]$$
где $\alpha_j = 0.1 + 0.05j$, где $j$ - порядковый номер в списке подгруппы.

$$x_i = \alpha_j + ih, \quad h = \frac{1}{n}, \quad i = 0,1,\dots,n$$
где $n = 10$.

$$f(x) = \alpha_j e^x + (1 - \alpha_j)\cos{x}$$

Точки восстановления: $x^* = x_0 + \frac{2}{3}h,\quad x^{**} = x_{n/2} + \frac{1}{2}h,\quad x^{***} = x_n - \frac{1}{3}h$.

Таким образом, для $j = 3$, $n = 10$ получим:

$$\alpha_j = 0.1 + 0.05 \cdot 3 =0.25$$

$$[a, b] = [0.25, 1.25]$$

$$h = 0.1 \qquad x_i = 0.25 + 0.1 \cdot i \qquad i = 0,1, \dots, 10$$

$$f(x) = 0.25 e^x + 0.75 \cos{x}$$

Точки восстановления: $x^* = 0.25 + \frac{1}{15} = \frac{19}{60}, \quad x^{**} = 0.25 + 0.05 = 0.75, \quad x^{***} = 1.25 - \frac{1}{30} = \frac{73}{60},$

где $x_0 = 0.25, \quad x_n = 1.25.$

Необходимо: 
1. Методом наименьших квадратов построить элемент наилучшего приближения $\Phi_0$ 5-й степени при $p(x) = 1$. 
2. Интерполяционный многочлен Лагранжа
3. Оценить невязку по формуле из метода Ньютона 
4. Построение интерполяционного многочлена Лагранжа с измененной сеткой узлов при помощи многочлена Чебышева

--- 
### **2. Методом наименьших квадратов построить элемент наилучшего приближения $\Phi_0$ 5-й степени при $p(x) = 1$.**

Искомый элемент $\Phi_0(x)$ представляется в виде линейной комбинации базисных функций $\varphi_i(x) = x^i$:
$$\Phi_0(x) = \sum_{i=0}^{5} c_i x^i$$

Коэффициенты $c_i$ определяются из условия ортогональности невязки подпространству:
$$(f - \Phi_0, \varphi_j) = 0, \quad j = 0,1,\dots,5$$

В случае таблично заданной функции скалярное произведение задаётся в дискретной форме:
$$(f,g) = \sum_{k=0}^{N} f(x_k) g(x_k)$$

Тогда коэффициенты $c_i$ находятся из системы уравнений:
$$\sum_{i=0}^{k} \left( \sum_{j=0}^{10} x_j^{i+k} \right) c_i = \sum_{j=0}^{10} f(x_j) x_j^k, \quad k = 0,1,\dots,5$$

Решение данной системы позволяет построить многочлен $\Phi_0(x)$. Для решения системы воспользуемся методом Гаусса.

In [ ]:
import numpy as np
import linear_algebra as linalg
import pandas as pd

def f(x: float) -> float:
    return 0.25 * np.exp(x) + 0.75*np.cos(x)

n = 10
alpha = 0.25
h = 1/n
K = 5

X = [alpha + i*h for i in range(n+1)]
F_x = [float(f(xi)) for xi in X]
recovery_x = (X[0] + h * 2/3, X[int(n/2)] + h * 1/2, X[n] - h * 1/3)

df = pd.DataFrame({
    "x": X,
    "f(x)": F_x
}).set_index("x").T
df 

x,0.25,0.35,0.45,0.55,0.65,0.75,0.85,0.95,1.05,1.15,1.25
f(x),1.047691,1.059296,1.067413,1.072707,1.075948,1.078017,1.079899,1.08269,1.087591,1.095914,1.109078


In [70]:
def gaussian_method(a: list[list])->list[float]:
    def change_system_for_select_main_element(system, j):
        max_row = max(system[j:], key=lambda row: row[j])
        max_index = system.index(max_row)
        system[max_index], system[j] = system[j], system[max_index]
        m = 1 if max_index == 0 else -1
        return system, m
    def straight_stroke(a):
        n = len(a)
        for k in range(0,n):
            a, _ = change_system_for_select_main_element(a,k)
            for j in range(n,k-1,-1):
                a[k][j] /= a[k][k]
            for i in range(k+1,n):
                for j in range(n,k-1,-1):
                    a[i][j] = a[i][j] - a[i][k]*a[k][j]
        return a
    def reverse_stroke(a: list[list[float]])->list[float]:
        n = len(a)
        x = [0.] * n
        x[n-1] = a[n-1][n]
        for i in range(n-2,-1,-1):
            s = 0
            for j in range(i+1,n):
                s+=a[i][j]*x[j]
            x[i] = a[i][n] - s
        return x
    
    return reverse_stroke(straight_stroke(a))

b = np.zeros(K+1)
A = np.zeros((K+1, K+2)) #K+2 для расшриенной матрицы

for row in range(K+1):
    b[row] = sum(F_x[j] * (X[j]**row) for j in range(n+1))
    A[row][-1] = b[row]
    for col in range (K+1):
        A[row][col] = sum(X[t]**(col+row) for t in range(n+1))

print("\n", "Вектор b:")
linalg.show_matrix([b])
print("Расширенная матрица A|b:")
linalg.show_matrix(A)
c = gaussian_method(A.tolist())
print("Коэффициенты c:")
linalg.show_matrix([c], e=True)
print(c)


 Вектор b:
 11.8562   8.9460   7.9356   7.7712   8.0871   8.7548
Расширенная матрица A|b:
 11.0000   8.2500   7.2875   7.1156   7.3888   7.9852  11.8562
  8.2500   7.2875   7.1156   7.3888   7.9852   8.8716   8.9460
  7.2875   7.1156   7.3888   7.9852   8.8716  10.0566   7.9356
  7.1156   7.3888   7.9852   8.8716  10.0566  11.5751   7.7712
  7.3888   7.9852   8.8716  10.0566  11.5751  13.4829   8.0871
  7.9852   8.8716  10.0566  11.5751  13.4829  15.8560   8.7548
Коэффициенты c:
1.0000e+00 2.4987e-01 -2.4935e-01 4.0017e-02 4.4013e-02 2.3911e-04
[1.0000108564660362, 0.24986724682201125, -0.2493507662598533, 0.040017287413288805, 0.044013441319538604, 0.00023911200378093586]


Таким образом, мы получили коэффициенты $c_i$ многочлена наилучшего среднеквадратического приближения $\Phi_0$:
$$\Phi_0(x) = 1.0000108564660362 + 0.24986724682201125x + -0.2493507662598533x^2 + 0.040017287413288805x^3 +$$
$$ + 0.044013441319538604x^4 + 0.00023911200378093586x^5$$

Для оценки погрешности вычисления полинома вычислим:

$$\Delta f = \left(\sum_{j=0}^{10} (\Phi_0(x_j) - f(x_j))^2 \right)^{\frac{1}{2}}$$

И вычислим невязку в точках восстановления:
$$r(x^*) = \Phi_0(x^*) - f (x^*)$$

In [65]:
def Phi(x: float, coeffs: list[float]) -> float:
    return sum(coeffs[i]*x**i for i in range(len(coeffs)))

inaccuracy = np.sqrt(sum(f(x) - Phi(x, c) for x in X)**2)
print ("Погрешность вычислений:", inaccuracy)

residuals =  [Phi(x, c) - f(x) for x in recovery_x]

df = pd.DataFrame({
    "Точки восстановления": recovery_x,
    "f(x)": [f(x) for x in recovery_x],
    "Phi(x)": [Phi(x, coeffs=c) for x in recovery_x],
    "Невязка": residuals
})
df = df.style.format({"Невязка": "{:.5e}"})
df


Погрешность вычислений: 2.220446049250313e-16


,Точки восстановления,f(x),Phi(x),Невязка
0,0.316667,1.055845,1.055845,-4.60907e-08
1,0.800000,1.078915,1.078915,8.32533e-09
2,1.216667,1.104060,1.104060,3.40238e-08


### **3. Интерполяционный многочлен Лагранжа**